In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
%sql
select * from fintech360.silver.bureau_data limit 5;

In [0]:
df = spark.read.table('fintech360.silver.bureau_data')

In [0]:
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. READ SOURCE TABLE
# ============================================================

source_df = spark.read.table("fintech360.silver.bureau_data")

df = (source_df
    .withColumn("_record_id", F.col("bureau_record_id").cast("long"))
    .withColumn("_customer_id", F.col("customer_id").cast("long"))
    .withColumn("_credit_score", F.col("credit_score").cast("int"))
    .withColumn("_active_accounts", F.col("active_accounts").cast("int"))
    .withColumn("_total_accounts", F.col("total_accounts").cast("int"))
    .withColumn("_outstanding", F.col("total_outstanding_amount").cast("double"))
    .withColumn("_overdue", F.col("overdue_amount").cast("double"))
    .withColumn("_enq_30d", F.col("enquiries_last_30d").cast("int"))
    .withColumn("_enq_90d", F.col("enquiries_last_90d").cast("int"))
    .withColumn("_delinquency", F.col("delinquency_ever").cast("int"))
    .withColumn("_written_off", F.col("written_off_accounts").cast("int"))
    .withColumn("_provider", F.upper(F.trim(F.col("bureau_provider"))))
    .withColumn("_score_band", F.upper(F.trim(F.col("score_band"))))
    .withColumn("_report_ts", F.col("report_ts").cast("long"))
    .withColumn("_created_at", F.col("created_at").cast("long"))
    .withColumn("_updated_at", F.col("updated_at").cast("long"))
)

# ============================================================
# 2. CONVERT TIMESTAMPS
# report_ts = Unix seconds
# created_at / updated_at = Unix milliseconds
# ============================================================

df = (df
    .withColumn(
        "_report_datetime",
        F.to_timestamp(F.from_unixtime(F.col("_report_ts")))
    )
    .withColumn(
        "_created_datetime",
        F.to_timestamp(
            F.from_unixtime(F.floor(F.col("_created_at") / 1000))
        )
    )
    .withColumn(
        "_updated_datetime",
        F.to_timestamp(
            F.from_unixtime(F.floor(F.col("_updated_at") / 1000))
        )
    )
)

# ============================================================
# 3. IDENTIFY DUPLICATE RECORDS
# Quarantine all rows sharing a duplicate key.
# ============================================================

record_window = Window.partitionBy("_record_id")
reference_window = Window.partitionBy("report_reference")

df = (df
    .withColumn(
        "_record_count",
        F.when(
            F.col("_record_id").isNotNull(),
            F.count("*").over(record_window)
        ).otherwise(F.lit(0))
    )
    .withColumn(
        "_reference_count",
        F.when(
            F.col("report_reference").isNotNull(),
            F.count("*").over(reference_window)
        ).otherwise(F.lit(0))
    )
)

# ============================================================
# 4. DEFINE DATA QUALITY RULES
# ============================================================

allowed_providers = [
    "SYNTHETIC_BUREAU_A",
    "SYNTHETIC_BUREAU_B",
    "SYNTHETIC_BUREAU_C"
]

allowed_score_bands = ["LOW", "MEDIUM", "HIGH", "VERY_HIGH"]

rules = [
    # Primary keys and references
    (
        F.col("bureau_record_id").isNull() |
        F.col("_record_id").isNull(),
        "INVALID_BUREAU_RECORD_ID"
    ),
    (
        F.col("customer_id").isNull() |
        F.col("_customer_id").isNull() |
        (F.col("_customer_id") <= 0),
        "INVALID_CUSTOMER_ID"
    ),
    (
        F.col("_record_count") > 1,
        "DUPLICATE_BUREAU_RECORD_ID"
    ),
    (
        F.col("report_reference").isNull() |
        (F.length(F.trim(F.col("report_reference"))) == 0),
        "MISSING_REPORT_REFERENCE"
    ),
    (
        F.col("_reference_count") > 1,
        "DUPLICATE_REPORT_REFERENCE"
    ),

    # Bureau provider and credit score
    (
        F.col("_provider").isNull() |
        (~F.col("_provider").isin(allowed_providers)),
        "INVALID_BUREAU_PROVIDER"
    ),
    (
        F.col("credit_score").isNull() |
        F.col("_credit_score").isNull() |
        (F.col("_credit_score") < 300) |
        (F.col("_credit_score") > 900),
        "INVALID_CREDIT_SCORE"
    ),
    (
        F.col("_score_band").isNull() |
        (~F.col("_score_band").isin(allowed_score_bands)),
        "INVALID_SCORE_BAND"
    ),

    # Account counts
    (
        F.col("active_accounts").isNull() |
        F.col("_active_accounts").isNull() |
        (F.col("_active_accounts") < 0),
        "INVALID_ACTIVE_ACCOUNTS"
    ),
    (
        F.col("total_accounts").isNull() |
        F.col("_total_accounts").isNull() |
        (F.col("_total_accounts") < 0),
        "INVALID_TOTAL_ACCOUNTS"
    ),
    (
        F.col("_active_accounts").isNotNull() &
        F.col("_total_accounts").isNotNull() &
        (F.col("_active_accounts") > F.col("_total_accounts")),
        "ACTIVE_ACCOUNTS_EXCEED_TOTAL"
    ),
    (
        F.col("written_off_accounts").isNull() |
        F.col("_written_off").isNull() |
        (F.col("_written_off") < 0),
        "INVALID_WRITTEN_OFF_ACCOUNTS"
    ),
    (
        F.col("_written_off").isNotNull() &
        F.col("_total_accounts").isNotNull() &
        (F.col("_written_off") > F.col("_total_accounts")),
        "WRITTEN_OFF_EXCEED_TOTAL_ACCOUNTS"
    ),

    # Outstanding and overdue amounts
    (
        F.col("total_outstanding_amount").isNull() |
        F.col("_outstanding").isNull() |
        F.isnan(F.col("_outstanding")) |
        (F.col("_outstanding") < 0),
        "INVALID_OUTSTANDING_AMOUNT"
    ),
    (
        F.col("overdue_amount").isNull() |
        F.col("_overdue").isNull() |
        F.isnan(F.col("_overdue")) |
        (F.col("_overdue") < 0),
        "INVALID_OVERDUE_AMOUNT"
    ),

    # Enquiry counts
    (
        F.col("enquiries_last_30d").isNull() |
        F.col("_enq_30d").isNull() |
        (F.col("_enq_30d") < 0),
        "INVALID_ENQUIRIES_30D"
    ),
    (
        F.col("enquiries_last_90d").isNull() |
        F.col("_enq_90d").isNull() |
        (F.col("_enq_90d") < 0),
        "INVALID_ENQUIRIES_90D"
    ),
    (
        F.col("_enq_30d").isNotNull() &
        F.col("_enq_90d").isNotNull() &
        (F.col("_enq_30d") > F.col("_enq_90d")),
        "ENQUIRIES_30D_EXCEED_90D"
    ),

    # Binary delinquency flag
    (
        F.col("delinquency_ever").isNull() |
        F.col("_delinquency").isNull() |
        (~F.col("_delinquency").isin([0, 1])),
        "INVALID_DELINQUENCY_FLAG"
    ),

    # Timestamp checks
    (
        F.col("report_ts").isNull() |
        F.col("_report_ts").isNull() |
        F.col("_report_datetime").isNull() |
        (F.col("_report_datetime") > F.current_timestamp()),
        "INVALID_REPORT_TIMESTAMP"
    ),
    (
        F.col("created_at").isNull() |
        F.col("_created_at").isNull() |
        F.col("_created_datetime").isNull(),
        "INVALID_CREATED_AT"
    ),
    (
        F.col("updated_at").isNull() |
        F.col("_updated_at").isNull() |
        F.col("_updated_datetime").isNull(),
        "INVALID_UPDATED_AT"
    ),
    (
        F.col("_created_datetime").isNotNull() &
        F.col("_updated_datetime").isNotNull() &
        (F.col("_updated_datetime") < F.col("_created_datetime")),
        "UPDATED_BEFORE_CREATED"
    )
]

# ============================================================
# 5. CREATE FAILURE REASONS
# ============================================================

df = df.withColumn(
    "dq_failed_reasons",
    F.concat_ws(
        " | ",
        *[
            F.when(condition, F.lit(reason))
            for condition, reason in rules
        ]
    )
)

# ============================================================
# 6. CREATE PASSED AND QUARANTINED DATAFRAMES
# ============================================================

helper_columns = [
    "_record_id", "_customer_id", "_credit_score",
    "_active_accounts", "_total_accounts", "_outstanding",
    "_overdue", "_enq_30d", "_enq_90d", "_delinquency",
    "_written_off", "_provider", "_score_band", "_report_ts",
    "_created_at", "_updated_at", "_report_datetime",
    "_created_datetime", "_updated_datetime",
    "_record_count", "_reference_count"
]

original_columns = source_df.columns

bureau_data_passed = (
    df.filter(F.col("dq_failed_reasons") == "")
      .select(*original_columns)
)

quarantine_data = (
    df.filter(F.col("dq_failed_reasons") != "")
      .drop(*helper_columns)
)

# ============================================================
# 7. DATA QUALITY SUMMARY
# ============================================================

source_count = source_df.count()
passed_count = bureau_data_passed.count()
failed_count = quarantine_data.count()

print(f"Source records      : {source_count}")
print(f"Passed records      : {passed_count}")
print(f"Quarantined records : {failed_count}")
print(f"Reconciled records  : {passed_count + failed_count}")

# Display failed records with reasons
display(
    quarantine_data.select(
        "bureau_record_id",
        "customer_id",
        "bureau_provider",
        "credit_score",
        "active_accounts",
        "total_accounts",
        "dq_failed_reasons"
    )
)

# Display passed records
display(bureau_data_passed)


In [0]:

# Create target schemas
spark.sql("CREATE SCHEMA IF NOT EXISTS fintech360.data_quality_check")
spark.sql("CREATE SCHEMA IF NOT EXISTS fintech360.quaratine_data")

# Store passed records
(bureau_data_passed.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.data_quality_check.bureau_data"))

# Store failed records with dq_failed_reasons
(quarantine_data.write
    .format("delta")
    .mode("append")
    .saveAsTable("fintech360.quarantine_data.bureau_data"))

print("Data Quality Gate completed successfully.")
